# Cycle 5 : Test oversampling


In [6]:
### Import des modules
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import utils
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
data = pd.read_csv('data/rafined/employees.csv', sep=',')

In [8]:
train_data = utils.split_train_data(data, 'a_quitte_l_entreprise')

In [9]:
preprocessor = utils.create_preprocessor()

In [ ]:
from sklearn.feature_selection import SelectFromModel
import numpy as np

feature_selector = SelectFromModel(
    estimator=RandomForestClassifier(n_estimators=50, random_state=42),
    max_features=20,
    threshold=-np.inf
)

final_model = RandomForestClassifier(
    max_depth=6,
    min_samples_leaf=15,
    random_state=42
)

In [22]:
from imblearn.over_sampling import SMOTE
from technova_correlation_cleaning import CorrelationFilter
from technova_features import TechNovaFeatureEngineering
from imblearn.pipeline import Pipeline

random_forest_model = RandomForestClassifier(random_state=42)
logistic_regression_model = LogisticRegression(random_state=42, max_iter=1000)

models = [
    {'name' : 'LogisticRegression', 'model': logistic_regression_model},
    {'name' : 'RandomForestClassifier', 'model': random_forest_model},
]

for model in models:
    print(f'Training model: {model["name"]}')

    pipeline = Pipeline([
        ('features', TechNovaFeatureEngineering()),
        ('corr_cleaning', CorrelationFilter(threshold=0.80)),
        ('preprocessor', preprocessor),
        ('feature_selection', feature_selector),
        ('sampling', SMOTE(random_state=42)),
        ('model', model['model']),
    ])

    utils.benchmark(pipeline, train_data)

    print(f'------------------------------\n')


Training model: LogisticRegression
--- Validation Fold Results ---
Validation Recall : [0.75       0.71794872 0.69230769 0.675      0.825     ], Recall moyen : 0.7320512820512821
Validation F1-Scores [0.54545455 0.51376147 0.49541284 0.46153846 0.55462185], F1 moyen : 0.5141578335318217
Validation ROC AUC : [0.84783163 0.77537938 0.8070382  0.78051282 0.84282051], ROC moyen : 0.8107165096807953

--- Train Fold Results (Overfit Check) ---
Train Recall : [0.76582278 0.74842767 0.78616352 0.77848101 0.78481013], Recall moyen : 0.7727410238038374
Train F1-Scores [0.53070175 0.53968254 0.54466231 0.56164384 0.53563715], F1 moyen : 0.5424655176162425
Train ROC AUC : [0.84020201 0.85533787 0.85516093 0.85543269 0.83701117], ROC moyen : 0.8486289355544672
------------------------------

Training model: RandomForestClassifier
--- Validation Fold Results ---
Validation Recall : [0.325      0.35897436 0.35897436 0.375      0.5       ], Recall moyen : 0.3835897435897436
Validation F1-Scores [0.4  